<a href="https://colab.research.google.com/github/Ethan-Brooke/APF-Paper-2-The-Structure-of-Admissible-Physics/blob/main/APF_Reviewer_Walkthrough.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Paper 2 — Structure
## Reviewer Walkthrough · Phase 22 Edition

[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.18604839.svg)](https://doi.org/10.5281/zenodo.18604839)

**Author:** E.S. Brooke · **Paper:** v5.4 main + v2.1 supplement · **Codebase:** v6.9

---

### What this notebook is

Paper 2 carries Paper 1's spine into the Standard Model's gauge structure and field content. Theorem R (non-closure), $T_M$ (interface monogamy), and $L_{\rm count}$ ($C_{\rm total} = 61$) live here.

### What this notebook is **not**

A derivation of fermion masses (Paper 4), a derivation of Hilbert space (Paper 5), or a derivation of cosmological parameters (Paper 6/8).

### Before you begin

If you are a cold AI agent or human reviewer new to APF, read these four files in `ai_context/` **first**:

1. **`ARGUMENT_FLOW.md`** — the one-page structural spine.
2. **`LOCAL_VS_IMPORTED.md`** — what this paper proves vs imports.
3. **`CLAIMS_LEDGER.md`** — row-by-row attack surface.
4. **`DO_NOT_CLAIM.md`** — predictable overclaims and how to avoid them.


## §1 · Setup

Clone the paper-companion repo and load the rendering helpers.

In [ ]:
!git clone -q https://github.com/Ethan-Brooke/APF-Paper-2-The-Structure-of-Admissible-Physics.git 2>/dev/null || true
%cd -q APF-Paper-2-The-Structure-of-Admissible-Physics
!pip install -q -e . 2>&1 | tail -2

In [ ]:
from apf.bank import REGISTRY, get_check, run_all
from apf.apf_utils import dag_reset, dag_put, dag_has, dag_get
from fractions import Fraction
from IPython.display import display, Markdown, Latex, HTML
import inspect

print(f'Bank registry loaded: {len(REGISTRY)} checks.')
print('This repo: 14 bank-registered checks bundled.')

### Phase 22 `show()` helper

Every check returns a result dict. `show()` runs the check, badges the status colour-coded by epistemic tag, and surfaces the mathematical content inline.

Colour code: 🟢 `[P]` proved from A1 · 🟡 `[P_structural]` conditional on upstream · 🟣 `[P_arith]` arithmetic identity · 🔵 `[P+lattice]` lattice QCD input · 🟠 `[C]` conjecture · 🔴 `[FAIL]` check did not pass.

In [ ]:

def _epistemic_badge(tag):
    colors = {
        'P': ('🟢', '#2ecc71', 'proved from A1'),
        'P_structural': ('🟡', '#f1c40f', 'conditional on upstream derivation'),
        'P_arith': ('🟣', '#9b59b6', 'arithmetic identity once formula chosen'),
        'P+lattice': ('🔵', '#3498db', 'proved using lattice QCD input'),
        'C': ('🟠', '#e67e22', 'conjecture; open, flagged'),
    }
    emoji, col, explain = colors.get(str(tag).strip('[]'), ('⚪', '#7f8c8d', 'unknown'))
    return f'<span style="background:{col}22;border:1px solid {col};border-radius:4px;padding:2px 8px;color:{col};font-weight:600">{emoji} {tag}</span> <em style="color:#666;font-size:0.9em">{explain}</em>'


def _render_value(v):
    if isinstance(v, Fraction):
        return f'$\\displaystyle \\frac{{{v.numerator}}}{{{v.denominator}}} = {float(v):.6f}$'
    if isinstance(v, dict) and all(isinstance(val, Fraction) for val in v.values()):
        items = [f'{k}={frac.numerator}/{frac.denominator}' for k, frac in v.items()]
        return '$' + ',\\ '.join(items) + '$'
    if isinstance(v, float):
        return f'`{v:.9g}`'
    if isinstance(v, (list, tuple)) and len(v) < 8:
        return '`' + ', '.join(str(x) for x in v) + '`'
    return f'`{v}`'


def show(check_name, *, run=True, verbose=True):
    try:
        check = get_check(check_name)
    except KeyError:
        display(Markdown(f'**❓ Check `{check_name}` not found.**'))
        return None

    display(Markdown(f'#### `{check_name}`'))
    doc = (check.__doc__ or '').strip()
    first_line = doc.split('\n')[0] if doc else '(no docstring)'
    display(Markdown(f'**Statement:** {first_line}'))

    if not run:
        return None

    try:
        result = check()
        passed = True
    except Exception as e:
        result = {'error': f'{type(e).__name__}: {e}'}
        passed = False

    tag = 'P'
    if isinstance(result, dict):
        for k in ('epistemic_status', 'epistemic', 'tag'):
            if k in result:
                tag = result[k]
                break
    if not passed:
        display(Markdown(f'**Status:** <span style="color:#e74c3c;font-weight:700">🔴 [FAIL]</span>'))
    else:
        display(Markdown(f'**Status:** {_epistemic_badge(tag)}'))

    if isinstance(result, dict):
        if 'key_result' in result:
            display(Markdown(f'**Key result:** {_render_value(result["key_result"])}'))
        if 'error' in result:
            display(Markdown(f'**Error:** `{result["error"]}`'))

        if verbose:
            skip = {'key_result', 'name', 'epistemic', 'epistemic_status', 'tag',
                    'dependencies', 'cross_refs', 'error', 'artifacts', 'statement',
                    'identity', 'consistent'}
            extra = {k: v for k, v in result.items() if k not in skip}
            if extra:
                rows = []
                for k, v in list(extra.items())[:10]:
                    rows.append(f'| `{k}` | {_render_value(v)} |')
                if rows:
                    display(Markdown('**Fields surfaced by the check:**\n\n| Field | Value |\n|---|---|\n' + '\n'.join(rows)))

        if 'dependencies' in result and result['dependencies']:
            deps = result['dependencies']
            if isinstance(deps, (list, tuple)):
                deps_str = ' · '.join(f'`{d}`' for d in deps)
                display(Markdown(f'**Depends on:** {deps_str}'))

    return result


print('show() helper loaded. Phase 22 gorgeous-math rendering enabled.')


## §2 · Non-closure: Theorem R

**Theorem R (Non-closure).** No admissibility framework can close on a single irreducible representation. Admissibility *forces* multiple irreducible components.

$$\boxed{\quad \text{Admissibility} \;+\; \text{Single-rep closure} \;\Rightarrow\; \bot \quad}$$

Why this matters: it rules out admissibility frameworks that would unify the SM into one simple gauge group via single-rep closure. Note: this does NOT rule out GUTs with multiple representations — see `DO_NOT_CLAIM.md` row 1.

In [ ]:
show('check_Theorem_R')

## §3 · Interface monogamy — $T_M$

**$T_M$ (Interface Monogamy).** An admissibility interface cannot be simultaneously responsible for two independent structural commitments.

Combined with Theorem R, $T_M$ blocks any attempt to collapse the SM gauge structure into a single interface.

In [ ]:
show('check_T_M')

## §4 · The gauge template — $SU(3) \times SU(2) \times U(1)$

Paper 1 proves $L_{\rm gauge\_template\_uniqueness}$; Paper 2 *uses* it to force $T_{\rm gauge}$: $N_c = 3$ by cost optimisation.

In [ ]:
show('check_T_gauge')

## §5 · Field content — $L_{\rm count}$

$$\boxed{\quad C_{\rm total} \;=\; \underbrace{45}_{\text{fermions: }3\text{ gens}\times15} \;+\; \underbrace{4}_{\text{Higgs doublet (complex)}} \;+\; \underbrace{12}_{\text{gauge: }8+3+1} \;=\; 61 \quad}$$

This is the $K$ of the ACC record (Paper 8). **Rigid.** No free parameters.

In [ ]:
show('check_L_count')

## §6 · I2 at the $L_{\rm count}$ seam (receiver from Paper 8)

Paper 2 Supplement v2.1 adds the integer-level statement of the I2 (gauge-cosmological) bridge identity:

$$\pi_F(\mathbf{ACC}_{\rm SM}) \;=\; 61 \;=\; \text{denom}\big(\pi_C(\mathbf{ACC}_{\rm SM})\big)\,.$$

The $K = 61$ that Paper 2's $L_{\rm count}$ produces is the SAME integer as the denominator Paper 8's $\pi_C$ uses. Full $I_2$ content (including uniqueness of the 42-dim $V_\Lambda$) is Paper 8 Theorem 1.1.

## §6 · Full bank pass

Run every check in this repo's bundled codebase subset.

In [ ]:
from apf.bank import run_all
from collections import Counter
results = run_all()
outcomes = Counter()
for name, res in results.items():
    if isinstance(res, dict) and 'error' in res:
        outcomes['ERROR'] += 1
    else:
        outcomes['PASS'] += 1
print(f'Total: {len(results)} checks')
for status, n in outcomes.most_common():
    print(f'  {status}: {n}')

### Where to go next

- **Paper PDF** — the main paper + Technical Supplement in this repo.
- **`ai_context/`** — the four audit-native files (ARGUMENT_FLOW, LOCAL_VS_IMPORTED, CLAIMS_LEDGER, DO_NOT_CLAIM).
- **[Canonical codebase v6.9](https://doi.org/10.5281/zenodo.18604548)** — the full 420-theorem bank.
- **[Paper 8 companion repo](https://github.com/Ethan-Brooke/APF-Paper-8-Admissibility-Capacity-Ledger)** — the pilot implementation of Phase 22 (anti-smuggling tests + minimal working example + full gorgeous-math Colab).

### Citation

```bibtex
@software{Brooke_Paper2_2026,
  author  = {Brooke, Ethan S.},
  title   = {The Structure of Admissible Physics},
  year    = 2026,
  version = {v5.4 main + v2.1 supplement},
  doi     = {10.5281/zenodo.18604839}
}
```

*Paper-companion repo · v5.4 main + v2.1 supplement · Phase 22 gorgeous-math edition · 2026-04-24.*